# 04 · Consensus map, contradictions, gaps, and the full CrewAI Flow

**Purpose.** Normalise dataset/metric/method names, group reported numbers by (task, dataset, metric), flag disagreements and have the Adjudicator explain them from the evidence alone, build the method-family x dataset matrix and have the Gap Analyst argue both sides of each candidate gap. Then run the whole Flow end to end on one topic, including the human approval checkpoint.

| | |
|---|---|
| **Inputs** | `models/scibert_relations/` (02), `data/extractions_*.jsonl` (03), OpenAlex |
| **Outputs** | `results/flow_runs/<topic>/` (review.md, records.json, citation_graph.html, contradictions.csv, gaps.csv, summary.json), `results/flow_summary.json` |
| **Expected runtime (Colab T4)** | 20-50 min |
| **Required secrets** | `GROQ_API_KEY`, `OPENALEX_MAILTO` |
| **GPU** | T4 recommended (NLI + SciBERT inference) |

Everything is cached under `/content/drive/MyDrive/litreview/` and every stage is
**resumable**: if Colab disconnects, re-run all cells and it continues where it
stopped. Only the cell marked CONFIG needs editing.


In [ ]:
# --- Colab setup: Drive, project code, secrets. Safe to re-run. ---
import json, os, subprocess, sys, zipfile
from pathlib import Path
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

ROOT = "/content/drive/MyDrive/litreview" if ON_COLAB else os.path.abspath("..")
os.makedirs(ROOT, exist_ok=True)
os.environ["LITREVIEW_ROOT"] = ROOT
REPO = os.path.join(ROOT, "repo")

# Optional: a git URL to clone the project from. Leave blank to use the folder or
# the zip on Drive instead.
REPO_URL = ""

def _find_project() -> str | None:
    """Locate the project code: an existing folder, a zip on Drive, or a clone."""
    if os.path.exists(os.path.join(REPO, "pyproject.toml")):
        return REPO
    # a litreview_*.zip uploaded anywhere under MyDrive/litreview
    zips = sorted(Path(ROOT).glob("litreview*.zip")) + sorted(Path(ROOT).glob("**/litreview*.zip"))
    for z in zips:
        print(f"extracting {z.name} ...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(ROOT)
        # the zip may contain a top-level folder; find the pyproject inside
        for cand in [REPO, ROOT] + [str(p.parent) for p in Path(ROOT).glob("*/pyproject.toml")]:
            if os.path.exists(os.path.join(cand, "pyproject.toml")):
                return cand
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=False)
        if os.path.exists(os.path.join(REPO, "pyproject.toml")):
            return REPO
    return None

if ON_COLAB:
    found = _find_project()
    if not found:
        raise SystemExit(
            "Could not find the project code. Do one of these, then re-run this cell:
"
            "  1. upload litreview_colab.zip into MyDrive/litreview/ (easiest), or
"
            "  2. copy the project folder to MyDrive/litreview/repo, or
"
            "  3. set REPO_URL above to a git URL.
"
            "The notebooks deliberately contain no second copy of the code.")
    REPO = found
    print("project code:", REPO)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", f"{REPO}[ml,eval]"],
                   check=True)

import litreview
from litreview.config import SETTINGS, paths, secret
P = paths().mkdirs()
print("data root:", P.root)
print("already present:", {f.name: f.stat().st_size // 1024 for f in sorted(P.data.glob("*.jsonl"))})


In [ ]:
# ======================= CONFIG (the only cell to edit) =======================
PROVIDER = "groq"
MODEL    = ""
TOPIC    = "faithfulness and hallucination evaluation in abstractive summarisation"
N_PAPERS = 15                # corpus size for the demonstration run
SEEDS    = []                # optional seed paper titles to snowball from
AUTO_APPROVE = False         # False shows the corpus table and asks
# ==============================================================================


In [ ]:
# Provider comes from one config. Keys are read from Colab secrets, never hard-coded.
import os
os.environ["LITREVIEW_PROVIDER"] = PROVIDER
if MODEL:
    os.environ["LITREVIEW_MODEL"] = MODEL

from litreview.config import Settings
from litreview.llm import install_call_logger, probe, usage_summary
import litreview.config as _cfg
_cfg.SETTINGS = Settings()                      # re-read after setting the env vars
import litreview.llm as _llm
_llm.SETTINGS = _cfg.SETTINGS

install_call_logger(P.results / "llm_calls.jsonl")
print(json.dumps(_cfg.SETTINGS.describe(), indent=1))
print(json.dumps(probe(_cfg.SETTINGS), indent=1))


### Normalisation

Three stages, cheapest first: the manual alias table checked into the repo
(`litreview/resources/aliases.json`), then fuzzy matching, then sentence
embeddings for the paraphrases fuzzy matching misses. Anything still unmatched
keeps a tidied version of its original surface form, so nothing is silently
dropped -- and the unmatched names are reported, which is how the alias file gets
extended.

In [ ]:
from litreview.normalise import Normaliser
n = Normaliser(use_embeddings=False)
for s, kind in [("cnn/dm", "datasets"), ("RougeL", "metrics"), ("bart-large", "methods"),
                ("hallucination detection", "tasks"), ("Zorblax benchmark", "datasets")]:
    print(f"{s:28s} {kind:9s} -> {n.normalise(s, kind)}")
print("\nmethod families:", {m: n.method_family(m) for m in ["bart", "scibert", "gpt-4", "crewai"]})


### The Flow

Ten steps: `collect_corpus` → **human approval** → `extract` → `verify_claims` →
`classify_relations` → `build_map` → `find_gaps` → `write_review` →
`verify_review` (one revision pass) → `export`. Everything is cached under the run
directory, so a Flow that dies mid-way continues on the next call.

`as_crewai_flow()` wires the same steps into a real `crewai.flow.Flow` with
`@start` / `@listen`; `run()` calls them directly, which is what the tests use.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from litreview.flow import LitReviewFlow
from litreview.relations.scibert import RelationClassifier

clf = None
if (P.models / "scibert_relations").exists():
    clf = RelationClassifier(P.models / "scibert_relations")
else:
    print("no classifier found; relations will be skipped (run notebook 02 first)")

def approve(table):
    display(pd.DataFrame(table))
    if AUTO_APPROVE:
        return True
    return (input("approve this corpus? [Y/n] ").strip().lower() or "y").startswith("y")

flow = LitReviewFlow(topic=TOPIC, n_papers=N_PAPERS, seeds=SEEDS, approve=approve,
                     classifier=clf, progress=lambda xs: tqdm(xs, desc="step"))
summary = flow.as_crewai_flow().kickoff()
if not isinstance(summary, dict):
    summary = flow.state.exports
summary = dict(summary) | {"usage": usage_summary()}
(P.results / "flow_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps({k: v for k, v in summary.items() if k != "usage"}, indent=2))


In [ ]:
from IPython.display import HTML, Markdown, display
print(flow.state.exports)
display(Markdown(flow.state.review.get("markdown", "(no review)")))


In [ ]:
cons = pd.DataFrame(flow.state.contradictions)
display(cons.head(10) if len(cons) else "no contradictions flagged")
gaps = pd.DataFrame(flow.state.gaps)
display(gaps[["kind", "method_family", "dataset", "verdict", "score",
              "argument_for", "argument_against"]].head(10) if len(gaps) else "no gaps")


In [ ]:
# the typed citation graph, inline
p = Path(flow.state.exports.get("graph_html", ""))
display(HTML(p.read_text()[:400_000]) if p.exists() else HTML("<i>no graph written</i>"))


### Next

**05_distill_extractor_qlora.ipynb** — distil the API Extractor into a small local
model, using the Verifier as the data filter. That notebook **requires** a GPU.